<a href="https://colab.research.google.com/github/haojin-teng/PS3/blob/main/ps3_starter_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [3]:
region = np.array(["West"] * 2_100 + list(rng.choice(["East", "North", "South"], size=N_CUSTOMERS - 2_100)))
rng.shuffle(region)
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)
customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": region,
    "revenue": np.maximum(REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS), REV_FLOOR).round(2),
    "signup_date": pd.Timestamp("2020-01-01") + pd.to_timedelta(rng.integers(0, 1_800, N_CUSTOMERS), unit="D"),
})
orders_df = pd.DataFrame({"customer_id": np.repeat(customers_df["customer_id"].to_numpy(), n_orders)})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")
print(con.sql("SELECT COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue FROM customers"))
print(con.sql("SELECT COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers_with_orders FROM orders"))

┌───────────┬─────────────┐
│ customers │ avg_revenue │
│   int64   │   double    │
├───────────┼─────────────┤
│      3000 │      813.03 │
└───────────┴─────────────┘

┌────────┬───────────────────────┐
│ orders │ customers_with_orders │
│ int64  │         int64         │
├────────┼───────────────────────┤
│  20568 │                  3000 │
└────────┴───────────────────────┘



### Step 1.2: Write both queries (7 pts)

In [4]:
JOINED_ROWS = "SELECT c.customer_id, c.revenue FROM customers c JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'"
PLAIN_ROWS = "SELECT customer_id, revenue FROM customers WHERE region = 'West'"

per_customer, n_plain = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({PLAIN_ROWS})").fetchone()
per_order, n_joined = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({JOINED_ROWS})").fetchone()
print(f"West, per customer: ${per_customer:.2f} over {n_plain:,} rows")
print(f"West, after the join: ${per_order:.2f} over {n_joined:,} rows")

West, per customer: $809.98 over 2,100 rows
West, after the join: $1240.07 over 14,265 rows


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*
The per-customer number is correct. Revenue is recorded once per customer, so the unit of observation is the customer; the join repeats each customer once per order, and since revenue rises with order count, the joined average is inflated.

### Step 1.3: The rule (10 pts)

In [5]:
def customer_grain(con, row_sql):
    """A warning string if the rows repeat customer_id, else None."""
    n_rows, n_keys = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM ({row_sql})").fetchone()
    if n_rows > n_keys:
        return f"FAN-OUT: {n_rows:,} rows for {n_keys:,} customers"
    return None

print("joined:", customer_grain(con, JOINED_ROWS))
print("plain: ", customer_grain(con, PLAIN_ROWS))

joined: FAN-OUT: 14,265 rows for 2,100 customers
plain:  None


*Your answer here.*
The check compares COUNT(*) with COUNT(DISTINCT customer_id). It fires on the joined query and returns None on the plain query.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [6]:
unknown_ids = rng.choice(customers_df["customer_id"].to_numpy(), size=450, replace=False)   # 15% of 3,000
con.register("unknown_ids", pd.DataFrame({"customer_id": unknown_ids}))
con.execute("UPDATE customers SET region = NULL WHERE customer_id IN (SELECT customer_id FROM unknown_ids)")
print(con.sql("SELECT COUNT(*) AS customers, COUNT(*) - COUNT(region) AS null_region FROM customers"))

┌───────────┬─────────────┐
│ customers │ null_region │
│   int64   │    int64    │
├───────────┼─────────────┤
│      3000 │         450 │
└───────────┴─────────────┘



### Step 2.2: Lose them, then find them (15 pts)

In [7]:
n_ne = con.execute("SELECT COUNT(*) FROM customers WHERE region != 'West'").fetchone()[0]
n_distinct = con.execute("SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'").fetchone()[0]
print(f"!= 'West': {n_ne:,} rows | IS DISTINCT FROM 'West': {n_distinct:,} rows | dropped: {n_distinct - n_ne:,}")
print(con.sql("SELECT NULL != 'West' AS null_ne, NULL IS DISTINCT FROM 'West' AS null_distinct"))

!= 'West': 752 rows | IS DISTINCT FROM 'West': 1,202 rows | dropped: 450
┌─────────┬───────────────┐
│ null_ne │ null_distinct │
│ boolean │    boolean    │
├─────────┼───────────────┤
│ NULL    │ true          │
└─────────┴───────────────┘



*Your answer here.*
`!= 'West'` returns 752 rows, `IS DISTINCT FROM 'West'` returns 1,202: the 450 NULL-region customers are dropped because `NULL != 'West'` is NULL, not true. If they are newer, smaller and churn more, every "West vs. the rest" comparison misstates the rest.

### Step 2.3: Aggregate without collapsing (10 pts)

In [8]:
WINDOW_SQL = """
    SELECT customer_id, region, revenue,
           AVG(revenue) OVER (PARTITION BY region) AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS vs_region
    FROM customers
    ORDER BY customer_id
"""
window_df = con.execute(WINDOW_SQL).df()
print(window_df.head(6).round(2).to_string(index=False))
print(f"rows in {N_CUSTOMERS:,}, rows out {len(window_df):,}")

 customer_id region  revenue  region_avg  vs_region
           1   West   442.00      804.01    -362.01
           2   West   842.33      804.01      38.32
           3   West  1200.59      804.01     396.58
           4   East  1088.37      818.85     269.52
           5   West   563.02      804.01    -240.99
           6  South   563.31      806.22    -242.91
rows in 3,000, rows out 3,000


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [9]:
TRUE_PROGRAMME = 50.0
n_accounts = 5_000
account_size = rng.normal(0, 1, n_accounts)
west = (rng.random(n_accounts) < 1 / (1 + np.exp(-1.5 * account_size))).astype(int)
revenue = 1000 + 300 * account_size + TRUE_PROGRAMME * west + rng.normal(0, 100, n_accounts)
prog = pd.DataFrame({"account_size": account_size, "west": west, "revenue": revenue})

raw = prog.loc[prog.west == 1, "revenue"].mean() - prog.loc[prog.west == 0, "revenue"].mean()
held = smf.ols("revenue ~ west + account_size", data=prog).fit().params["west"]
print(f"truth {TRUE_PROGRAMME:.2f} | raw gap {raw:.2f} | holding account size {held:.2f}")

truth 50.00 | raw gap 373.01 | holding account size 51.46


*Your answer here.*
Truth \$50. Raw gap \$373, mostly account size: bigger accounts were more likely to be in West. Holding account size fixed gives \$51.

### Step 3.2: The control that ruins it (13 pts)

In [10]:
prog["tickets"] = rng.poisson(np.maximum(0, 5 + 3 * west + 0.02 * (revenue - 1000)))   # caused by the programme AND by revenue
worse = smf.ols("revenue ~ west + account_size + tickets", data=prog).fit().params["west"]
print(f"truth {TRUE_PROGRAMME:.2f} | holding account size {held:.2f} | adding tickets {worse:.2f}")

truth 50.00 | holding account size 51.46 | adding tickets 1.11


*Your answer here.*
Tickets rise with both the programme and revenue, so customers with the same ticket count are not comparable: a non-programme customer needs high revenue to get there, a programme customer does not. Holding tickets fixed hides the effect. Control only for things fixed before the programme, like account size.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [11]:
"""
review_query(con, sql) -> list[ReviewWarning]

Pre-dashboard review of a single DuckDB SELECT. Dependencies: duckdb, pandas.

Checks
  JOIN_ROW_COUNT_CHANGE  a join returns more (or fewer) rows than its driving table
  NULLABLE_FILTER        a WHERE predicate on a nullable column silently drops unknowns
  FANOUT_AGGREGATE       SUM/AVG/COUNT read a table whose rows the join has repeated

Severity
  error    changes the number this query returns on the data as it is now
  warning  the schema allows the problem, or it is present but may be intended
  info     the check could not be completed (so an empty list never means "not checked")

The SQL is parsed by DuckDB itself (json_serialize_sql), not by regex, and the
statistics are computed with small GROUP BY queries, never by running the query.
Tables and registered DataFrames must be visible to `con` by name.
"""
from __future__ import annotations

import copy
import json
from dataclasses import dataclass, field, asdict

import duckdb
import pandas as pd

SAFE_AGGS = {"min", "max", "any_value", "first", "last", "arg_min", "arg_max", "bool_and", "bool_or"}
FANOUT_AGGS = {"sum", "avg", "mean", "count", "count_star", "string_agg", "list", "median", "stddev",
               "stddev_samp", "var_samp", "quantile_cont"}
NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "COMPARE_DISTINCT_FROM",
                    "COMPARE_NOT_DISTINCT_FROM", "OPERATOR_COALESCE"}
NULL_AWARE_FUNCS = {"coalesce", "ifnull", "nvl", "isnull", "notnull"}
NEGATED_TYPES = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN", "OPERATOR_NOT"}
NEGATED_FUNCS = {"!~~", "!~~*", "not_like", "not_ilike"}


@dataclass(frozen=True)
class ReviewWarning:
    code: str
    severity: str
    message: str
    details: dict = field(default_factory=dict)


def warnings_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    """One row per warning, for asserting in tests or showing in a notebook."""
    return pd.DataFrame([asdict(w) for w in warnings], columns=["code", "severity", "message", "details"])


# --------------------------------------------------------------------------- parsing helpers

def _walk(node):
    """Yield every dict in a parse tree."""
    if isinstance(node, dict):
        yield node
        for v in node.values():
            yield from _walk(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk(v)


def _colrefs(node):
    return [n["column_names"] for n in _walk(node) if n.get("class") == "COLUMN_REF"]


def _qident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def _parse(con, sql):
    out = json.loads(con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0])
    if out.get("error"):
        raise ValueError(out.get("error_message", "could not parse SQL"))
    return out


class _Scope:
    """Tables in the FROM clause: alias -> table name, columns, nullability."""

    def __init__(self, con):
        self.con = con
        self.alias_to_table: dict[str, str] = {}
        self.columns: dict[str, dict[str, bool]] = {}   # alias -> {column: schema_nullable}
        self.from_nodes: dict[str, dict] = {}

    def add(self, node):
        name = node["table_name"]
        alias = node.get("alias") or name
        desc = self.con.execute(f"DESCRIBE SELECT * FROM {_qident(name)}").fetchdf()
        self.alias_to_table[alias] = name
        self.columns[alias] = {r.column_name: str(r.null).upper() != "NO" for r in desc.itertuples()}
        self.from_nodes[alias] = node

    def resolve(self, column_names):
        """['c', 'region'] or ['region'] -> ('c', 'region'), or None if ambiguous/unknown."""
        if len(column_names) >= 2 and column_names[-2] in self.columns:
            return column_names[-2], column_names[-1]
        owners = [a for a, cols in self.columns.items() if column_names[-1] in cols]
        return (owners[0], column_names[-1]) if len(owners) == 1 else None


def _conjuncts(expr):
    if expr is None:
        return []
    if expr.get("class") == "CONJUNCTION" and expr.get("type") == "CONJUNCTION_AND":
        return [c for child in expr["children"] for c in _conjuncts(child)]
    return [expr]


def _filtered_sql(con, scope, alias, conjuncts):
    """SELECT * FROM <table> AS <alias> WHERE <the WHERE conjuncts that touch only this table>.

    Built by asking DuckDB to serialize a template statement, attaching the
    conjunct sub-trees, and deserializing, so no SQL text is pasted together."""
    table = scope.alias_to_table[alias]
    template = _parse(con, f"SELECT * FROM {_qident(table)} AS {_qident(alias)}")
    if conjuncts:
        cond = conjuncts[0] if len(conjuncts) == 1 else {
            "class": "CONJUNCTION", "type": "CONJUNCTION_AND", "alias": "", "query_location": 0,
            "children": conjuncts}
        template["statements"][0]["node"]["where_clause"] = copy.deepcopy(cond)
    return con.execute("SELECT json_deserialize_sql(?)", [json.dumps(template)]).fetchone()[0]


def _join_edges(node, scope, edges):
    """Flatten the FROM tree into base tables and join edges (left alias, right alias, keys, type)."""
    t = node.get("type")
    if t == "BASE_TABLE":
        scope.add(node)
        return [node.get("alias") or node["table_name"]]
    if t != "JOIN":
        raise NotImplementedError(f"FROM item of type {t}")
    left = _join_edges(node["left"], scope, edges)
    right = _join_edges(node["right"], scope, edges)
    keys = []
    if node.get("using_columns"):
        for col in node["using_columns"]:
            la = next(a for a in left if col in scope.columns[a])
            ra = next(a for a in right if col in scope.columns[a])
            keys.append(((la, col), (ra, col)))
    for c in _conjuncts(node.get("condition")):
        if c.get("type") == "COMPARE_EQUAL" and c["left"].get("class") == c["right"].get("class") == "COLUMN_REF":
            a, b = scope.resolve(c["left"]["column_names"]), scope.resolve(c["right"]["column_names"])
            if a and b and a[0] in left and b[0] in right:
                keys.append((a, b))
            elif a and b and b[0] in left and a[0] in right:
                keys.append((b, a))
    edges.append(dict(left_aliases=left, right_aliases=right, keys=keys,
                      join_type=node.get("join_type", "INNER")))
    return left + right


# --------------------------------------------------------------------------- main entry point

def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[ReviewWarning]:
    out: list[ReviewWarning] = []
    try:
        parsed = _parse(con, sql)
    except Exception as e:  # noqa: BLE001
        return [ReviewWarning("PARSE_ERROR", "info", f"Could not parse: {e}")]
    stmts = parsed.get("statements", [])
    if len(stmts) != 1 or stmts[0].get("node", {}).get("type") != "SELECT_NODE":
        return [ReviewWarning("NOT_A_SIMPLE_SELECT", "info",
                              "Only a single SELECT (no UNION/CTE chain at top level) is reviewed.")]
    node = stmts[0]["node"]

    scope, edges = _Scope(con), []
    try:
        _join_edges(node["from_table"], scope, edges)
    except Exception as e:  # noqa: BLE001
        return [ReviewWarning("FROM_UNSUPPORTED", "info",
                              f"FROM clause not reviewed ({e}); subqueries and table functions are skipped.")]

    where = _conjuncts(node.get("where_clause"))
    # Conjuncts that mention exactly one table can be applied to that table alone.
    pushable: dict[str, list] = {a: [] for a in scope.columns}
    for c in where:
        owners = {r[0] for r in map(scope.resolve, _colrefs(c)) if r}
        if len(owners) == 1:
            pushable[owners.pop()].append(c)
    filtered = {a: _filtered_sql(con, scope, a, pushable[a]) for a in scope.columns}

    outer_aliases = set()
    for e in edges:
        if e["join_type"] == "LEFT":
            outer_aliases.update(e["right_aliases"])
        elif e["join_type"] == "RIGHT":
            outer_aliases.update(e["left_aliases"])
        elif e["join_type"] in ("OUTER", "FULL"):
            outer_aliases.update(e["left_aliases"] + e["right_aliases"])

    repeated = _check_joins(con, scope, edges, filtered, out)
    _check_where(con, scope, where, outer_aliases, out)
    _check_aggregates(con, scope, node, edges, filtered, repeated, out)
    return out


# --------------------------------------------------------------------------- check 1: join row count

def _check_joins(con, scope, edges, filtered, out):
    """Return {alias: edge_stats} for every table whose rows the joins repeat.

    HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN TWICE (or even once):
    an equi-join's output size depends only on how many rows share each key
    value. Collapse each side to a key histogram, nL(k) and nR(k), with one
    GROUP BY scan per table. Then

        rows out (INNER) = sum over matching k of nL(k) * nR(k)
        rows out (LEFT)  = the above + left rows whose key has no match

    The histograms have one row per distinct key, so joining them is cheap.
    The same histograms say who gets repeated: a left row with key k appears
    nR(k) times, so if max nR(k) > 1 the left table is fanned out (and vice
    versa). The WHERE conjuncts that touch only one table are applied before
    building its histogram, so the counts match the rows the query really sees.
    """
    repeated = {}
    for e in edges:
        # With 3+ tables each edge is measured on its own; compounding across edges is not modelled.
        approx = len(e["left_aliases"]) + len(e["right_aliases"]) > 2
        if not e["keys"]:
            out.append(ReviewWarning("JOIN_NOT_CHECKED", "info",
                                     "Join has no equality key between two columns (cross/theta join?); "
                                     "row count not checked.", dict(join_type=e["join_type"])))
            continue
        la, ra = e["keys"][0][0][0], e["keys"][0][1][0]
        lk = ", ".join(f"{_qident(la)}.{_qident(l[1])} AS k{i}" for i, (l, _) in enumerate(e["keys"]))
        rk = ", ".join(f"{_qident(ra)}.{_qident(r[1])} AS k{i}" for i, (_, r) in enumerate(e["keys"]))
        ks = ", ".join(f"k{i}" for i in range(len(e["keys"])))
        stats = con.execute(f"""
            WITH l AS (SELECT {lk}, COUNT(*) AS n FROM ({filtered[la]}) AS {_qident(la)} GROUP BY ALL),
                 r AS (SELECT {rk}, COUNT(*) AS n FROM ({filtered[ra]}) AS {_qident(ra)} GROUP BY ALL),
                 m AS (SELECT l.n AS nl, r.n AS nr FROM l JOIN r USING ({ks}))
            SELECT (SELECT COALESCE(SUM(n), 0) FROM l),
                   (SELECT COALESCE(SUM(n), 0) FROM r),
                   (SELECT COALESCE(SUM(nl * nr), 0) FROM m),
                   (SELECT COALESCE(SUM(nl), 0) FROM m),
                   (SELECT COALESCE(SUM(nr), 0) FROM m),
                   (SELECT COALESCE(MAX(nr), 0) FROM m),
                   (SELECT COALESCE(MAX(nl), 0) FROM m)
        """).fetchone()
        n_left, n_right, inner, left_matched, right_matched, max_nr, max_nl = map(int, stats)
        jt = e["join_type"]
        n_out = inner
        if jt in ("LEFT", "OUTER", "FULL"):
            n_out += n_left - left_matched
        if jt in ("RIGHT", "OUTER", "FULL"):
            n_out += n_right - right_matched
        driving = n_right if jt == "RIGHT" else n_left
        details = dict(left=scope.alias_to_table[la], right=scope.alias_to_table[ra], join_type=jt,
                       keys=[f"{l[0]}.{l[1]} = {r[0]}.{r[1]}" for l, r in e["keys"]],
                       driving_rows=driving, rows_out=n_out,
                       factor=round(n_out / driving, 3) if driving else None,
                       max_repeats_of_left_row=max_nr, max_repeats_of_right_row=max_nl,
                       approximate=approx)
        if n_out != driving:
            out.append(ReviewWarning(
                "JOIN_ROW_COUNT_CHANGE", "error",
                f"Join {details['left']} -> {details['right']} turns {driving:,} rows into {n_out:,} "
                f"({details['factor']}x).", details))
        if max_nr > 1:
            repeated[la] = dict(edge=e, partner=ra, details=details)
        if max_nl > 1:
            repeated[ra] = dict(edge=e, partner=la, details=details)
    return repeated


# --------------------------------------------------------------------------- check 2: nullable filters

def _check_where(con, scope, where, outer_aliases, out):
    """Flag WHERE conjuncts that drop rows whose column is NULL without saying so.

    WHY THIS IS A CORRECTNESS ISSUE, NOT STYLE: SQL uses three-valued logic.
    Any comparison with NULL (NULL = 'West', NULL != 'West', NULL < 5,
    NULL IN (...)) evaluates to UNKNOWN, not TRUE or FALSE, and NOT UNKNOWN is
    still UNKNOWN. WHERE keeps only rows that are TRUE. So `region != 'West'`
    does not mean "everyone except West"; it means "everyone whose region is
    known and is not West". The rows vanish with no error, the query still
    runs, and "West" + "not West" no longer adds up to the total. Rewriting
    the query cannot fix that unless the author decides what an unknown
    region should count as; that decision changes the number, so it is a
    question about the result, not about how the SQL looks.

    A conjunct is treated as deliberate if it already mentions NULL handling:
    IS [NOT] NULL, IS [NOT] DISTINCT FROM, or COALESCE/IFNULL.
    On the outer side of a LEFT JOIN the same predicate also discards the
    unmatched rows the LEFT JOIN was written to keep, i.e. it becomes an
    INNER JOIN.
    """
    for c in where:
        if any(n.get("type") in NULL_AWARE_TYPES or str(n.get("function_name", "")).lower() in NULL_AWARE_FUNCS
               for n in _walk(c)):
            continue
        negated = any(n.get("type") in NEGATED_TYPES or n.get("function_name") in NEGATED_FUNCS
                      for n in _walk(c))
        for ref in {tuple(r) for r in _colrefs(c)}:
            res = scope.resolve(list(ref))
            if not res:
                continue
            alias, col = res
            outer = alias in outer_aliases
            schema_nullable = scope.columns[alias][col]
            if not (schema_nullable or outer):
                continue
            table = scope.alias_to_table[alias]
            n_null = int(con.execute(
                f"SELECT COUNT(*) - COUNT({_qident(col)}) FROM {_qident(table)}").fetchone()[0])
            details = dict(table=table, column=col, null_rows_in_table=n_null, negated=negated,
                           outer_side_of_left_join=outer, schema_nullable=schema_nullable)
            if outer:
                sev, msg = "error", (f"Filter on {alias}.{col} (outer side of a LEFT JOIN) removes the "
                                     "unmatched rows, so the LEFT JOIN behaves as an INNER JOIN.")
            elif n_null and negated:
                sev, msg = "error", (f"Negated filter on {table}.{col} silently drops {n_null:,} rows where "
                                     f"{col} IS NULL; they are in neither group. Use IS DISTINCT FROM or "
                                     f"add OR {col} IS NULL if they belong in the result.")
            elif n_null:
                sev, msg = "warning", (f"Filter on {table}.{col} excludes {n_null:,} rows where {col} IS NULL. "
                                       "Correct only if 'unknown' should never count as a match.")
            else:
                sev, msg = "info", (f"{table}.{col} is nullable but has no NULLs today; this filter will "
                                    "start dropping rows silently when unknowns appear.")
            out.append(ReviewWarning("NULLABLE_FILTER", sev, msg, details))


# --------------------------------------------------------------------------- check 3: aggregates on fanned rows

def _check_aggregates(con, scope, node, edges, filtered, repeated, out):
    """An aggregate is fanned out if its argument comes from a table the join repeats.

    MIN/MAX and COUNT(DISTINCT ...) are unaffected by duplicates and pass.
    SUM, AVG and COUNT are not. For a bare column argument the size of the
    error is computed without the join: each row of the repeated table is
    weighted by m(k), its partner's key count (the same histogram as above).

        true value   = AVG(v)                     over one row per entity
        fanned value = SUM(v * m) / SUM(m)        what the query will return
    """
    targets = node.get("select_list", []) + [node.get("having")] if node.get("having") else node.get("select_list", [])
    for fn in (n for n in _walk(targets) if n.get("class") in ("FUNCTION", "AGGREGATE")):
        name = str(fn.get("function_name", "")).lower()
        if name in SAFE_AGGS or name not in FANOUT_AGGS or fn.get("distinct"):
            continue
        refs = [scope.resolve(r) for r in _colrefs(fn.get("children", []))]
        hit = sorted({r[0] for r in refs if r and r[0] in repeated})
        if name == "count_star" and repeated:
            hit = sorted(repeated)
        if not hit:
            continue
        details = dict(function=name, repeated_tables=[scope.alias_to_table[a] for a in hit])
        quant = _quantify(con, scope, fn, name, repeated, filtered) if len(hit) == 1 else None
        if quant:
            details.update(quant)
        msg = (f"{name.upper()} reads {', '.join(details['repeated_tables'])}, whose rows the join repeats; "
               "each entity is weighted by its number of matches.")
        if quant and "overstatement" in quant:
            msg += f" Value {quant['fanned_value']:,.2f} vs {quant['one_row_per_entity_value']:,.2f}."
        out.append(ReviewWarning("FANOUT_AGGREGATE", "error", msg, details))


def _quantify(con, scope, fn, name, repeated, filtered):
    children = fn.get("children", [])
    if name not in ("sum", "avg", "mean") or len(children) != 1 or children[0].get("class") != "COLUMN_REF":
        return None
    alias, col = scope.resolve(children[0]["column_names"]) or (None, None)
    if alias not in repeated:
        return None
    rep = repeated[alias]
    e, partner = rep["edge"], rep["partner"]
    own_keys = [(l if l[0] == alias else r) for l, r in e["keys"]]
    par_keys = [(r if l[0] == alias else l) for l, r in e["keys"]]
    ks = ", ".join(f"k{i}" for i in range(len(own_keys)))
    tk = ", ".join(f"{_qident(alias)}.{_qident(k[1])} AS k{i}" for i, k in enumerate(own_keys))
    pk = ", ".join(f"{_qident(partner)}.{_qident(k[1])} AS k{i}" for i, k in enumerate(par_keys))
    outer_keeps = (e["join_type"] == "LEFT" and alias in e["left_aliases"]) or \
                  (e["join_type"] == "RIGHT" and alias in e["right_aliases"])
    join = "LEFT JOIN" if outer_keeps else "JOIN"
    row = con.execute(f"""
        WITH p AS (SELECT {pk}, COUNT(*) AS m FROM ({filtered[partner]}) AS {_qident(partner)} GROUP BY ALL),
             t AS (SELECT {tk}, {_qident(alias)}.{_qident(col)} AS v
                   FROM ({filtered[alias]}) AS {_qident(alias)})
        SELECT AVG(t.v), SUM(t.v), SUM(t.v * COALESCE(p.m, 1)) / SUM(COALESCE(p.m, 1)),
               SUM(t.v * COALESCE(p.m, 1)), COUNT(*), SUM(COALESCE(p.m, 1))
        FROM t {join} p USING ({ks})
    """).fetchone()
    avg_true, sum_true, avg_fan, sum_fan, n_ent, n_rows = row
    if name == "sum":
        true, fan = float(sum_true), float(sum_fan)
    else:
        true, fan = float(avg_true), float(avg_fan)
    return dict(column=f"{alias}.{col}", entities=int(n_ent), rows_aggregated=int(n_rows),
                one_row_per_entity_value=round(true, 2), fanned_value=round(fan, 2),
                overstatement=round(fan - true, 2),
                overstatement_pct=round(100 * (fan - true) / true, 1) if true else None)

In [12]:
# Run review_query on your Phase 1 and Phase 2 queries.
def _check_where(con, scope, where, outer_aliases, out):
    """Same as the model's version, with str() added so dict-valued "type" keys do not raise TypeError."""
    for c in where:
        if any(str(n.get("type")) in NULL_AWARE_TYPES or str(n.get("function_name", "")).lower() in NULL_AWARE_FUNCS
               for n in _walk(c)):
            continue
        negated = any(str(n.get("type")) in NEGATED_TYPES or str(n.get("function_name")) in NEGATED_FUNCS
                      for n in _walk(c))
        for ref in {tuple(r) for r in _colrefs(c)}:
            res = scope.resolve(list(ref))
            if not res:
                continue
            alias, col = res
            outer = alias in outer_aliases
            schema_nullable = scope.columns[alias][col]
            if not (schema_nullable or outer):
                continue
            table = scope.alias_to_table[alias]
            n_null = int(con.execute(
                f"SELECT COUNT(*) - COUNT({_qident(col)}) FROM {_qident(table)}").fetchone()[0])
            details = dict(table=table, column=col, null_rows_in_table=n_null, negated=negated,
                           outer_side_of_left_join=outer, schema_nullable=schema_nullable)
            if outer:
                sev, msg = "error", (f"Filter on {alias}.{col} (outer side of a LEFT JOIN) removes the "
                                     "unmatched rows, so the LEFT JOIN behaves as an INNER JOIN.")
            elif n_null and negated:
                sev, msg = "error", (f"Negated filter on {table}.{col} silently drops {n_null:,} rows where "
                                     f"{col} IS NULL; they are in neither group. Use IS DISTINCT FROM or "
                                     f"add OR {col} IS NULL if they belong in the result.")
            elif n_null:
                sev, msg = "warning", (f"Filter on {table}.{col} excludes {n_null:,} rows where {col} IS NULL. "
                                       "Correct only if 'unknown' should never count as a match.")
            else:
                sev, msg = "info", (f"{table}.{col} is nullable but has no NULLs today; this filter will "
                                    "start dropping rows silently when unknowns appear.")
            out.append(ReviewWarning("NULLABLE_FILTER", sev, msg, details))

In [13]:
queries = {
    "Phase 1 low": "SELECT AVG(revenue), COUNT(*) FROM customers WHERE region = 'West'",
    "Phase 1 high": "SELECT AVG(c.revenue), COUNT(*) FROM customers c JOIN orders o ON o.customer_id = c.customer_id WHERE c.region = 'West'",
    "Phase 2 !=": "SELECT COUNT(*) FROM customers WHERE region != 'West'",
    "Phase 2 IS DISTINCT FROM": "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'",
    "Phase 2.3 window": WINDOW_SQL,
}
for name, q in queries.items():
    ws = review_query(con, q)
    print(f"\n{name}: {len(ws)} warning(s)")
    for w in ws:
        print(f"  [{w.severity}] {w.code}: {w.message}")


Phase 1 low: 1 warning(s)
  [warning] NULLABLE_FILTER: Filter on customers.region excludes 450 rows where region IS NULL. Correct only if 'unknown' should never count as a match.

Phase 1 high: 4 warning(s)
  [error] JOIN_ROW_COUNT_CHANGE: Join customers -> orders turns 1,798 rows into 12,077 (6.717x).
  [warning] NULLABLE_FILTER: Filter on customers.region excludes 450 rows where region IS NULL. Correct only if 'unknown' should never count as a match.
  [error] FANOUT_AGGREGATE: AVG reads customers, whose rows the join repeats; each entity is weighted by its number of matches. Value 1,240.52 vs 804.01.
  [error] FANOUT_AGGREGATE: COUNT_STAR reads customers, whose rows the join repeats; each entity is weighted by its number of matches.

Phase 2 !=: 1 warning(s)
  [error] NULLABLE_FILTER: Negated filter on customers.region silently drops 450 rows where region IS NULL; they are in neither group. Use IS DISTINCT FROM or add OR region IS NULL if they belong in the result.

Phase 2 IS DIST

*Your answer here.*
- Caught: fan-out in the joined query (1,798 → 12,077 rows; \$1,240.52 vs \$804.01) and the `!=` filter dropping 450 NULLs.
- Missed: crashed until _check_where was patched; skips FROM-subqueries and NULL partitions; cannot judge the unit of observation or the Phase 3 tickets problem.
- Caught the \$1,240? Yes.

### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*
Director,

The correct West average is \$810 per customer. The \$1,240 counts each customer once per order, and high-order customers have higher revenue.

The programme works but is small: the raw \$373 gap is mostly account size; adjusted, it is \$51 (truth \$50). Controlling for support tickets wrongly drops it to \$1.

Before any slide: check row counts after joins, handle NULLs explicitly, control only for pre-programme variables, and run the query reviewer.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Claude Opus 5.5, 2026-11-01

*Prompt:*
```
[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.
```

*Reply:*
I've written `review_query.py` (the function) and `run_phase1_review.py` (which rebuilds your Phase 1 warehouse with seed 3916 and runs both queries through it, with assertions).

**How it works.** DuckDB parses the SQL itself through `json_serialize_sql`, so nothing is pattern-matched on text. Each check returns `ReviewWarning(code, severity, message, details)` objects, and `warnings_frame()` turns them into a DataFrame. Severity is `error` when the number the query returns right now is affected, `warning` when the schema or data allows the problem but it may be intended, and `info` when a check couldn't run, so an empty list never means "not checked." The two explanations you asked for are in the docstrings of `_check_joins` (fan-out from key histograms, with no join executed) and `_check_where` (three-valued logic).

**Important caveat:** DuckDB isn't installed in my sandbox and there's no network, so I couldn't execute the function. Both files compile. To check the figures below, I rebuilt the same data in pandas with the same seed and computed the arithmetic the function performs: the row counts and the order-weighted average. The likeliest point of failure is a parse-tree key name that differs in your DuckDB version, such as `using_columns`, `column_names`, or the `json_deserialize_sql` round-trip. If `run_phase1_review.py` fails, paste the traceback.

**Evaluate: would it have caught the \$1,240?**

Yes, for the joined query.

- **`q_high`** (customers JOIN orders, AVG(c.revenue), West) returns two errors.
  - `JOIN_ROW_COUNT_CHANGE`: the join turns 2,100 West customers into 13,928 rows (6.63x). One customer is repeated 72 times.
  - `FANOUT_AGGREGATE` on `AVG(c.revenue)`: the query returns \$1,194.20, while one row per customer gives \$798.60. The overstatement is \$395.60 (+49.5%).
  - `COUNT(*)` is also flagged as counting orders, not customers.
- **`q_low`** returns no errors. The only flag is an `info` that `region` is nullable but has no NULLs yet.

The \$1,240 is the problem set's theoretical target; this seed produces \$1,194.20. The function's overstatement figure is the same weighted-average gap that produced it. Run as a gate before publishing, it would have stopped `q_high`, and the warning explains why: each customer is weighted by their order count, and revenue rises with order count.

**What it would not have done:**

- It doesn't prove `q_low` is correct. It only shows that `q_low` has no fan-out. Deciding that the customer is the unit of observation is still your call.
- It only helps if it actually runs before the dashboard.
- After Phase 2 introduces NULL regions, `region = 'West'` becomes a `warning` with a count of excluded rows. A `region != 'West'` filter becomes an `error`, because those customers end up in neither group.

```python
"""
review_query(con, sql) -> list[ReviewWarning]

Pre-dashboard review of a single DuckDB SELECT. Dependencies: duckdb, pandas.

Checks
  JOIN_ROW_COUNT_CHANGE  a join returns more (or fewer) rows than its driving table
  NULLABLE_FILTER        a WHERE predicate on a nullable column silently drops unknowns
  FANOUT_AGGREGATE       SUM/AVG/COUNT read a table whose rows the join has repeated

Severity
  error    changes the number this query returns on the data as it is now
  warning  the schema allows the problem, or it is present but may be intended
  info     the check could not be completed (so an empty list never means "not checked")

The SQL is parsed by DuckDB itself (json_serialize_sql), not by regex, and the
statistics are computed with small GROUP BY queries, never by running the query.
Tables and registered DataFrames must be visible to `con` by name.
"""
from __future__ import annotations

import copy
import json
from dataclasses import dataclass, field, asdict

import duckdb
import pandas as pd

SAFE_AGGS = {"min", "max", "any_value", "first", "last", "arg_min", "arg_max", "bool_and", "bool_or"}
FANOUT_AGGS = {"sum", "avg", "mean", "count", "count_star", "string_agg", "list", "median", "stddev",
               "stddev_samp", "var_samp", "quantile_cont"}
NULL_AWARE_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "COMPARE_DISTINCT_FROM",
                    "COMPARE_NOT_DISTINCT_FROM", "OPERATOR_COALESCE"}
NULL_AWARE_FUNCS = {"coalesce", "ifnull", "nvl", "isnull", "notnull"}
NEGATED_TYPES = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN", "OPERATOR_NOT"}
NEGATED_FUNCS = {"!~~", "!~~*", "not_like", "not_ilike"}


@dataclass(frozen=True)
class ReviewWarning:
    code: str
    severity: str
    message: str
    details: dict = field(default_factory=dict)


def warnings_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    """One row per warning, for asserting in tests or showing in a notebook."""
    return pd.DataFrame([asdict(w) for w in warnings], columns=["code", "severity", "message", "details"])


# --------------------------------------------------------------------------- parsing helpers

def _walk(node):
    """Yield every dict in a parse tree."""
    if isinstance(node, dict):
        yield node
        for v in node.values():
            yield from _walk(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk(v)


def _colrefs(node):
    return [n["column_names"] for n in _walk(node) if n.get("class") == "COLUMN_REF"]


def _qident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def _parse(con, sql):
    out = json.loads(con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0])
    if out.get("error"):
        raise ValueError(out.get("error_message", "could not parse SQL"))
    return out


class _Scope:
    """Tables in the FROM clause: alias -> table name, columns, nullability."""

    def __init__(self, con):
        self.con = con
        self.alias_to_table: dict[str, str] = {}
        self.columns: dict[str, dict[str, bool]] = {}   # alias -> {column: schema_nullable}
        self.from_nodes: dict[str, dict] = {}

    def add(self, node):
        name = node["table_name"]
        alias = node.get("alias") or name
        desc = self.con.execute(f"DESCRIBE SELECT * FROM {_qident(name)}").fetchdf()
        self.alias_to_table[alias] = name
        self.columns[alias] = {r.column_name: str(r.null).upper() != "NO" for r in desc.itertuples()}
        self.from_nodes[alias] = node

    def resolve(self, column_names):
        """['c', 'region'] or ['region'] -> ('c', 'region'), or None if ambiguous/unknown."""
        if len(column_names) >= 2 and column_names[-2] in self.columns:
            return column_names[-2], column_names[-1]
        owners = [a for a, cols in self.columns.items() if column_names[-1] in cols]
        return (owners[0], column_names[-1]) if len(owners) == 1 else None


def _conjuncts(expr):
    if expr is None:
        return []
    if expr.get("class") == "CONJUNCTION" and expr.get("type") == "CONJUNCTION_AND":
        return [c for child in expr["children"] for c in _conjuncts(child)]
    return [expr]


def _filtered_sql(con, scope, alias, conjuncts):
    """SELECT * FROM <table> AS <alias> WHERE <the WHERE conjuncts that touch only this table>.

    Built by asking DuckDB to serialize a template statement, attaching the
    conjunct sub-trees, and deserializing, so no SQL text is pasted together."""
    table = scope.alias_to_table[alias]
    template = _parse(con, f"SELECT * FROM {_qident(table)} AS {_qident(alias)}")
    if conjuncts:
        cond = conjuncts[0] if len(conjuncts) == 1 else {
            "class": "CONJUNCTION", "type": "CONJUNCTION_AND", "alias": "", "query_location": 0,
            "children": conjuncts}
        template["statements"][0]["node"]["where_clause"] = copy.deepcopy(cond)
    return con.execute("SELECT json_deserialize_sql(?)", [json.dumps(template)]).fetchone()[0]


def _join_edges(node, scope, edges):
    """Flatten the FROM tree into base tables and join edges (left alias, right alias, keys, type)."""
    t = node.get("type")
    if t == "BASE_TABLE":
        scope.add(node)
        return [node.get("alias") or node["table_name"]]
    if t != "JOIN":
        raise NotImplementedError(f"FROM item of type {t}")
    left = _join_edges(node["left"], scope, edges)
    right = _join_edges(node["right"], scope, edges)
    keys = []
    if node.get("using_columns"):
        for col in node["using_columns"]:
            la = next(a for a in left if col in scope.columns[a])
            ra = next(a for a in right if col in scope.columns[a])
            keys.append(((la, col), (ra, col)))
    for c in _conjuncts(node.get("condition")):
        if c.get("type") == "COMPARE_EQUAL" and c["left"].get("class") == c["right"].get("class") == "COLUMN_REF":
            a, b = scope.resolve(c["left"]["column_names"]), scope.resolve(c["right"]["column_names"])
            if a and b and a[0] in left and b[0] in right:
                keys.append((a, b))
            elif a and b and b[0] in left and a[0] in right:
                keys.append((b, a))
    edges.append(dict(left_aliases=left, right_aliases=right, keys=keys,
                      join_type=node.get("join_type", "INNER")))
    return left + right


# --------------------------------------------------------------------------- main entry point

def review_query(con: duckdb.DuckDBPyConnection, sql: str) -> list[ReviewWarning]:
    out: list[ReviewWarning] = []
    try:
        parsed = _parse(con, sql)
    except Exception as e:  # noqa: BLE001
        return [ReviewWarning("PARSE_ERROR", "info", f"Could not parse: {e}")]
    stmts = parsed.get("statements", [])
    if len(stmts) != 1 or stmts[0].get("node", {}).get("type") != "SELECT_NODE":
        return [ReviewWarning("NOT_A_SIMPLE_SELECT", "info",
                              "Only a single SELECT (no UNION/CTE chain at top level) is reviewed.")]
    node = stmts[0]["node"]

    scope, edges = _Scope(con), []
    try:
        _join_edges(node["from_table"], scope, edges)
    except Exception as e:  # noqa: BLE001
        return [ReviewWarning("FROM_UNSUPPORTED", "info",
                              f"FROM clause not reviewed ({e}); subqueries and table functions are skipped.")]

    where = _conjuncts(node.get("where_clause"))
    # Conjuncts that mention exactly one table can be applied to that table alone.
    pushable: dict[str, list] = {a: [] for a in scope.columns}
    for c in where:
        owners = {r[0] for r in map(scope.resolve, _colrefs(c)) if r}
        if len(owners) == 1:
            pushable[owners.pop()].append(c)
    filtered = {a: _filtered_sql(con, scope, a, pushable[a]) for a in scope.columns}

    outer_aliases = set()
    for e in edges:
        if e["join_type"] == "LEFT":
            outer_aliases.update(e["right_aliases"])
        elif e["join_type"] == "RIGHT":
            outer_aliases.update(e["left_aliases"])
        elif e["join_type"] in ("OUTER", "FULL"):
            outer_aliases.update(e["left_aliases"] + e["right_aliases"])

    repeated = _check_joins(con, scope, edges, filtered, out)
    _check_where(con, scope, where, outer_aliases, out)
    _check_aggregates(con, scope, node, edges, filtered, repeated, out)
    return out


# --------------------------------------------------------------------------- check 1: join row count

def _check_joins(con, scope, edges, filtered, out):
    """Return {alias: edge_stats} for every table whose rows the joins repeat.

    HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN TWICE (or even once):
    an equi-join's output size depends only on how many rows share each key
    value. Collapse each side to a key histogram, nL(k) and nR(k), with one
    GROUP BY scan per table. Then

        rows out (INNER) = sum over matching k of nL(k) * nR(k)
        rows out (LEFT)  = the above + left rows whose key has no match

    The histograms have one row per distinct key, so joining them is cheap.
    The same histograms say who gets repeated: a left row with key k appears
    nR(k) times, so if max nR(k) > 1 the left table is fanned out (and vice
    versa). The WHERE conjuncts that touch only one table are applied before
    building its histogram, so the counts match the rows the query really sees.
    """
    repeated = {}
    for e in edges:
        # With 3+ tables each edge is measured on its own; compounding across edges is not modelled.
        approx = len(e["left_aliases"]) + len(e["right_aliases"]) > 2
        if not e["keys"]:
            out.append(ReviewWarning("JOIN_NOT_CHECKED", "info",
                                     "Join has no equality key between two columns (cross/theta join?); "
                                     "row count not checked.", dict(join_type=e["join_type"])))
            continue
        la, ra = e["keys"][0][0][0], e["keys"][0][1][0]
        lk = ", ".join(f"{_qident(la)}.{_qident(l[1])} AS k{i}" for i, (l, _) in enumerate(e["keys"]))
        rk = ", ".join(f"{_qident(ra)}.{_qident(r[1])} AS k{i}" for i, (_, r) in enumerate(e["keys"]))
        ks = ", ".join(f"k{i}" for i in range(len(e["keys"])))
        stats = con.execute(f"""
            WITH l AS (SELECT {lk}, COUNT(*) AS n FROM ({filtered[la]}) AS {_qident(la)} GROUP BY ALL),
                 r AS (SELECT {rk}, COUNT(*) AS n FROM ({filtered[ra]}) AS {_qident(ra)} GROUP BY ALL),
                 m AS (SELECT l.n AS nl, r.n AS nr FROM l JOIN r USING ({ks}))
            SELECT (SELECT COALESCE(SUM(n), 0) FROM l),
                   (SELECT COALESCE(SUM(n), 0) FROM r),
                   (SELECT COALESCE(SUM(nl * nr), 0) FROM m),
                   (SELECT COALESCE(SUM(nl), 0) FROM m),
                   (SELECT COALESCE(SUM(nr), 0) FROM m),
                   (SELECT COALESCE(MAX(nr), 0) FROM m),
                   (SELECT COALESCE(MAX(nl), 0) FROM m)
        """).fetchone()
        n_left, n_right, inner, left_matched, right_matched, max_nr, max_nl = map(int, stats)
        jt = e["join_type"]
        n_out = inner
        if jt in ("LEFT", "OUTER", "FULL"):
            n_out += n_left - left_matched
        if jt in ("RIGHT", "OUTER", "FULL"):
            n_out += n_right - right_matched
        driving = n_right if jt == "RIGHT" else n_left
        details = dict(left=scope.alias_to_table[la], right=scope.alias_to_table[ra], join_type=jt,
                       keys=[f"{l[0]}.{l[1]} = {r[0]}.{r[1]}" for l, r in e["keys"]],
                       driving_rows=driving, rows_out=n_out,
                       factor=round(n_out / driving, 3) if driving else None,
                       max_repeats_of_left_row=max_nr, max_repeats_of_right_row=max_nl,
                       approximate=approx)
        if n_out != driving:
            out.append(ReviewWarning(
                "JOIN_ROW_COUNT_CHANGE", "error",
                f"Join {details['left']} -> {details['right']} turns {driving:,} rows into {n_out:,} "
                f"({details['factor']}x).", details))
        if max_nr > 1:
            repeated[la] = dict(edge=e, partner=ra, details=details)
        if max_nl > 1:
            repeated[ra] = dict(edge=e, partner=la, details=details)
    return repeated


# --------------------------------------------------------------------------- check 2: nullable filters

def _check_where(con, scope, where, outer_aliases, out):
    """Flag WHERE conjuncts that drop rows whose column is NULL without saying so.

    WHY THIS IS A CORRECTNESS ISSUE, NOT STYLE: SQL uses three-valued logic.
    Any comparison with NULL (NULL = 'West', NULL != 'West', NULL < 5,
    NULL IN (...)) evaluates to UNKNOWN, not TRUE or FALSE, and NOT UNKNOWN is
    still UNKNOWN. WHERE keeps only rows that are TRUE. So `region != 'West'`
    does not mean "everyone except West"; it means "everyone whose region is
    known and is not West". The rows vanish with no error, the query still
    runs, and "West" + "not West" no longer adds up to the total. Rewriting
    the query cannot fix that unless the author decides what an unknown
    region should count as; that decision changes the number, so it is a
    question about the result, not about how the SQL looks.

    A conjunct is treated as deliberate if it already mentions NULL handling:
    IS [NOT] NULL, IS [NOT] DISTINCT FROM, or COALESCE/IFNULL.
    On the outer side of a LEFT JOIN the same predicate also discards the
    unmatched rows the LEFT JOIN was written to keep, i.e. it becomes an
    INNER JOIN.
    """
    for c in where:
        if any(n.get("type") in NULL_AWARE_TYPES or str(n.get("function_name", "")).lower() in NULL_AWARE_FUNCS
               for n in _walk(c)):
            continue
        negated = any(n.get("type") in NEGATED_TYPES or n.get("function_name") in NEGATED_FUNCS
                      for n in _walk(c))
        for ref in {tuple(r) for r in _colrefs(c)}:
            res = scope.resolve(list(ref))
            if not res:
                continue
            alias, col = res
            outer = alias in outer_aliases
            schema_nullable = scope.columns[alias][col]
            if not (schema_nullable or outer):
                continue
            table = scope.alias_to_table[alias]
            n_null = int(con.execute(
                f"SELECT COUNT(*) - COUNT({_qident(col)}) FROM {_qident(table)}").fetchone()[0])
            details = dict(table=table, column=col, null_rows_in_table=n_null, negated=negated,
                           outer_side_of_left_join=outer, schema_nullable=schema_nullable)
            if outer:
                sev, msg = "error", (f"Filter on {alias}.{col} (outer side of a LEFT JOIN) removes the "
                                     "unmatched rows, so the LEFT JOIN behaves as an INNER JOIN.")
            elif n_null and negated:
                sev, msg = "error", (f"Negated filter on {table}.{col} silently drops {n_null:,} rows where "
                                     f"{col} IS NULL; they are in neither group. Use IS DISTINCT FROM or "
                                     f"add OR {col} IS NULL if they belong in the result.")
            elif n_null:
                sev, msg = "warning", (f"Filter on {table}.{col} excludes {n_null:,} rows where {col} IS NULL. "
                                       "Correct only if 'unknown' should never count as a match.")
            else:
                sev, msg = "info", (f"{table}.{col} is nullable but has no NULLs today; this filter will "
                                    "start dropping rows silently when unknowns appear.")
            out.append(ReviewWarning("NULLABLE_FILTER", sev, msg, details))


# --------------------------------------------------------------------------- check 3: aggregates on fanned rows

def _check_aggregates(con, scope, node, edges, filtered, repeated, out):
    """An aggregate is fanned out if its argument comes from a table the join repeats.

    MIN/MAX and COUNT(DISTINCT ...) are unaffected by duplicates and pass.
    SUM, AVG and COUNT are not. For a bare column argument the size of the
    error is computed without the join: each row of the repeated table is
    weighted by m(k), its partner's key count (the same histogram as above).

        true value   = AVG(v)                     over one row per entity
        fanned value = SUM(v * m) / SUM(m)        what the query will return
    """
    targets = node.get("select_list", []) + [node.get("having")] if node.get("having") else node.get("select_list", [])
    for fn in (n for n in _walk(targets) if n.get("class") in ("FUNCTION", "AGGREGATE")):
        name = str(fn.get("function_name", "")).lower()
        if name in SAFE_AGGS or name not in FANOUT_AGGS or fn.get("distinct"):
            continue
        refs = [scope.resolve(r) for r in _colrefs(fn.get("children", []))]
        hit = sorted({r[0] for r in refs if r and r[0] in repeated})
        if name == "count_star" and repeated:
            hit = sorted(repeated)
        if not hit:
            continue
        details = dict(function=name, repeated_tables=[scope.alias_to_table[a] for a in hit])
        quant = _quantify(con, scope, fn, name, repeated, filtered) if len(hit) == 1 else None
        if quant:
            details.update(quant)
        msg = (f"{name.upper()} reads {', '.join(details['repeated_tables'])}, whose rows the join repeats; "
               "each entity is weighted by its number of matches.")
        if quant and "overstatement" in quant:
            msg += f" Value {quant['fanned_value']:,.2f} vs {quant['one_row_per_entity_value']:,.2f}."
        out.append(ReviewWarning("FANOUT_AGGREGATE", "error", msg, details))


def _quantify(con, scope, fn, name, repeated, filtered):
    children = fn.get("children", [])
    if name not in ("sum", "avg", "mean") or len(children) != 1 or children[0].get("class") != "COLUMN_REF":
        return None
    alias, col = scope.resolve(children[0]["column_names"]) or (None, None)
    if alias not in repeated:
        return None
    rep = repeated[alias]
    e, partner = rep["edge"], rep["partner"]
    own_keys = [(l if l[0] == alias else r) for l, r in e["keys"]]
    par_keys = [(r if l[0] == alias else l) for l, r in e["keys"]]
    ks = ", ".join(f"k{i}" for i in range(len(own_keys)))
    tk = ", ".join(f"{_qident(alias)}.{_qident(k[1])} AS k{i}" for i, k in enumerate(own_keys))
    pk = ", ".join(f"{_qident(partner)}.{_qident(k[1])} AS k{i}" for i, k in enumerate(par_keys))
    outer_keeps = (e["join_type"] == "LEFT" and alias in e["left_aliases"]) or \
                  (e["join_type"] == "RIGHT" and alias in e["right_aliases"])
    join = "LEFT JOIN" if outer_keeps else "JOIN"
    row = con.execute(f"""
        WITH p AS (SELECT {pk}, COUNT(*) AS m FROM ({filtered[partner]}) AS {_qident(partner)} GROUP BY ALL),
             t AS (SELECT {tk}, {_qident(alias)}.{_qident(col)} AS v
                   FROM ({filtered[alias]}) AS {_qident(alias)})
        SELECT AVG(t.v), SUM(t.v), SUM(t.v * COALESCE(p.m, 1)) / SUM(COALESCE(p.m, 1)),
               SUM(t.v * COALESCE(p.m, 1)), COUNT(*), SUM(COALESCE(p.m, 1))
        FROM t {join} p USING ({ks})
    """).fetchone()
    avg_true, sum_true, avg_fan, sum_fan, n_ent, n_rows = row
    if name == "sum":
        true, fan = float(sum_true), float(sum_fan)
    else:
        true, fan = float(avg_true), float(avg_fan)
    return dict(column=f"{alias}.{col}", entities=int(n_ent), rows_aggregated=int(n_rows),
                one_row_per_entity_value=round(true, 2), fanned_value=round(fan, 2),
                overstatement=round(fan - true, 2),
                overstatement_pct=round(100 * (fan - true) / true, 1) if true else None)
```

```python
"""Run review_query against both Phase 1 queries of PS3 (seed 3916)."""
import numpy as np, pandas as pd, duckdb
from review_query import review_query, warnings_frame

rng = np.random.default_rng(3916)
N = 3000
region = np.array(['West'] * 2100 + list(rng.choice(['East', 'North', 'South'], N - 2100)))
rng.shuffle(region)
n_orders = 1 + rng.negative_binomial(0.8, 0.12, N)
revenue = np.maximum(60 * n_orders + rng.normal(400, 150, N), 50)
signup = pd.Timestamp('2020-01-01') + pd.to_timedelta(rng.integers(0, 1800, N), unit='D')
customers_df = pd.DataFrame(dict(customer_id=np.arange(1, N + 1), region=region,
                                 revenue=revenue.round(2), signup_date=signup))
cid = np.repeat(customers_df.customer_id.values, n_orders)
orders_df = pd.DataFrame(dict(order_id=np.arange(1, len(cid) + 1), customer_id=cid))

con = duckdb.connect()
con.register('customers_df', customers_df); con.register('orders_df', orders_df)
con.execute("CREATE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE TABLE orders AS SELECT * FROM orders_df")

q_low = "SELECT COUNT(*) AS n_rows, AVG(revenue) AS avg_revenue FROM customers WHERE region = 'West'"
q_high = """SELECT COUNT(*) AS n_rows, AVG(c.revenue) AS avg_revenue
FROM customers c JOIN orders o USING (customer_id)
WHERE c.region = 'West'"""

pd.set_option('display.max_colwidth', None)
for name, q in [("q_low", q_low), ("q_high", q_high)]:
    print(f"\n=== {name}: returns {con.sql(q).fetchone()}")
    w = review_query(con, q)
    print(warnings_frame(w)[["code", "severity", "message"]].to_string(index=False))
    for x in w:
        print("  ", x.code, x.details)

# Minimal assertions so this doubles as a test.
low, high = review_query(con, q_low), review_query(con, q_high)
assert not [w for w in low if w.severity == "error"]
codes = {w.code for w in high if w.severity == "error"}
assert {"JOIN_ROW_COUNT_CHANGE", "FANOUT_AGGREGATE"} <= codes
jc = next(w for w in high if w.code == "JOIN_ROW_COUNT_CHANGE").details
assert (jc["driving_rows"], jc["rows_out"]) == (2100, 13928), jc
fa = next(w for w in high if w.code == "FANOUT_AGGREGATE" and w.details.get("function") == "avg").details
assert abs(fa["fanned_value"] - con.sql(q_high).fetchone()[1]) < 0.01
assert abs(fa["one_row_per_entity_value"] - con.sql(q_low).fetchone()[1]) < 0.01
print("\nAll assertions passed.")
```

*Changes I made:*
- Ran it on my notebook data (seed 42) instead of the reply's seed-3916 rebuild; rewrote the queries without FROM-subqueries, which the function does not parse.
- Redefined _check_where with str() around n.get("type"): the parse tree also uses "type" for dict-valued type info, which raised TypeError: unhashable type: 'dict'.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.